In [ ]:
from pathlib import Path

import numpy as np
from PIL import Image

PATCH = 350
H0, W0 = 966, 1296  # sugar-beets rgb 
N_ROWS = (H0 + PATCH - 1) // PATCH  # 3
N_COLS = (W0 + PATCH - 1) // PATCH  # 4
CANVAS_H = N_ROWS * PATCH  # 1050
CANVAS_W = N_COLS * PATCH  # 1400
PAD_BOTTOM = CANVAS_H - H0
PAD_RIGHT = CANVAS_W - W0

print(
    f": {N_ROWS}×{N_COLS} = {N_ROWS * N_COLS} patches / ; "
    f" {CANVAS_H}×{CANVAS_W}; pad={PAD_BOTTOM}, ={PAD_RIGHT}"
)

In [ ]:
def find_repo(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / "EWS" / "data" / "sugar-beets-2016-DatasetNinja" / "ds" / "img").is_dir():
            return p
    raise FileNotFoundError(
        "。 AR-SSL4M ， cwd 。"
    )


REPO = find_repo(Path.cwd())
IMG_DIR = REPO / "EWS" / "data" / "sugar-beets-2016-DatasetNinja" / "ds" / "img"
OUT_DIR = REPO / "EWS" / "data" / "pretrain" / "npy"
if not OUT_DIR.is_dir():
    raise FileNotFoundError(
        f"（）， .npy : {OUT_DIR}"
    )

print("REPO:", REPO)
print("IMG_DIR:", IMG_DIR)
print("OUT_DIR:", OUT_DIR)

In [ ]:
def image_to_padded_canvas(arr_hwc: np.ndarray) -> np.ndarray:
    """arr: (H0, W0, C) uint8，，/。"""
    if arr_hwc.shape[0] != H0 or arr_hwc.shape[1] != W0:
        raise ValueError(f" H,W = {H0},{W0},  {arr_hwc.shape[:2]}")
    return np.pad(
        arr_hwc,
        ((0, PAD_BOTTOM), (0, PAD_RIGHT), (0, 0)),
        mode="constant",
        constant_values=0,
    )


def save_patches_for_image(img_path: Path) -> list[Path]:
    stem = img_path.stem
    im = Image.open(img_path).convert("RGB")
    arr = np.asarray(im, dtype=np.uint8)
    canvas = image_to_padded_canvas(arr)
    saved = []
    for r in range(N_ROWS):
        for c in range(N_COLS):
            y0, y1 = r * PATCH, (r + 1) * PATCH
            x0, x1 = c * PATCH, (c + 1) * PATCH
            patch = canvas[y0:y1, x0:x1, :].astype(np.float32) / 255.0
            out_path = OUT_DIR / f"{stem}__r{r}_c{c}.npy"
            np.save(out_path, patch, allow_pickle=False)
            saved.append(out_path)
    return saved

In [ ]:
pip install tqdm

In [ ]:
from tqdm.auto import tqdm

rgb_paths = sorted(IMG_DIR.glob("rgb_*.png"))
print(f" {len(rgb_paths)}  rgb ， {N_ROWS * N_COLS}  npy")

all_saved = []
for i, p in enumerate(tqdm(rgb_paths, desc=" sugar-beets", unit="")):
    paths = save_patches_for_image(p)
    all_saved.extend(paths)
    if (i + 1) % 100 == 0:
        tqdm.write(
            f" {i + 1} （ {p.name}）， {len(all_saved)}  npy"
        )

print(f"， {len(all_saved)}  .npy（ {OUT_DIR} ）")